### Pipeline parameters

`run_id` and `pipeline_name` are injected automatically when this notebook is invoked by the Data Factory pipeline. 

In [ ]:
run_id = ""  
pipeline_name = ""

### Configuration

Define the source table (`bronze.SampleData`, which is the Cosmos DB mirrored Delta table in the `bronze` schema) and the three silver output tables written to the `silver` schema.

The `metastore_functions` item is a **Fabric User Data Function** that exposes helpers for data-quality logging, dataset profiling, and transform lineage tracking.

In [ ]:

# Table references
BRONZE_TABLE = "bronze.SampleData"
SILVER_PRODUCTS = "silver.products"
SILVER_REVIEWS = "silver.reviews"
SILVER_PRICE_HISTORY = "silver.price_history"

# Initialize metastore User Data Functions (no pip install needed — deps managed in UDF item)
metastore = notebookutils.udf.getFunctions('PipelineMetadata')

### Create silver schema

Create the `silver` schema if it does not already exist. All silver tables are written to this schema.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

### Read bronze data and split by document type

Load the full bronze Delta table that Cosmos DB mirroring syncs. The table contains **two document types** in a single container: `products` and `reviews`. 

We split them into separate DataFrames using the `docType` discriminator field. Counts are printed to give a quick sanity check before downstream transformations.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

bronze_df = spark.read.table(BRONZE_TABLE)

# Separate products from reviews using docType discriminator
products_raw = bronze_df.filter(F.col("docType") == "product")
reviews_raw = bronze_df.filter(F.col("docType") == "review")

print(f"Bronze total: {bronze_df.count()}")
print(f"Products: {products_raw.count()}")
print(f"Reviews: {reviews_raw.count()}")

### Build silver_products

**Parse:** The `priceHistory` column is a nested JSON array, but Cosmos DB mirroring stores it as a plain string in the Delta table. We use `from_json` with an explicit schema (`array<struct<date,price>>`) to convert it into a queryable Spark array.

**Validate:** Drop rows missing `product_id`, `product_name`, or `current_price`. Reject prices ≤ 0 or ≥ 100,000 as data-quality guardrails.

**Enrich** with four derived columns:
- `days_since_available` — age of the product listing
- `price_change_pct` — percentage change between the first historical price and the current price
- `price_change_direction` — `"up"` (> 5 %), `"down"` (< −5 %), or `"stable"`
- `inventory_status` — `"low"` (< 100 units), `"medium"` (< 500), or `"high"`

In [ ]:

# Schema for the priceHistory JSON string — Cosmos DB mirror stores nested
# arrays as strings in the Delta table, so we must parse them explicitly.
price_history_schema = "array<struct<date:string,price:double>>"

silver_products = (
    products_raw
    .select(
        F.col("id").alias("product_id"),
        F.col("name").alias("product_name"),
        F.col("description"),
        F.col("categoryName").alias("category_name"),
        F.col("countryOfOrigin").alias("country_of_origin"),
        F.col("inventory").cast("int"),
        F.to_timestamp("firstAvailable").alias("first_available"),
        F.col("currentPrice").cast("double").alias("current_price"),
        # Parse the JSON string into a proper array<struct> so we can access elements
        F.from_json(F.col("priceHistory"), price_history_schema).alias("price_history_raw"),
    )
    # Validate: non-null required fields
    .filter(
        F.col("product_id").isNotNull()
        & F.col("product_name").isNotNull()
        & F.col("current_price").isNotNull()
        & (F.col("current_price") > 0)
        & (F.col("current_price") < 100000)  # Sanity check
    )
    # Enrich: compute derived fields
    .withColumn(
        "days_since_available",
        F.datediff(F.current_date(), F.col("first_available"))
    )
    .withColumn(
        "first_price",
        F.col("price_history_raw").getItem(0).getField("price")
    )
    .withColumn(
        "price_change_pct",
        F.round(
            ((F.col("current_price") - F.col("first_price")) / F.col("first_price")) * 100,
            2,
        ),
    )
    .withColumn(
        "price_change_direction",
        F.when(F.col("price_change_pct") > 5, "up")
        .when(F.col("price_change_pct") < -5, "down")
        .otherwise("stable"),
    )
    .withColumn(
        "inventory_status",
        F.when(F.col("inventory") < 100, "low")
        .when(F.col("inventory") < 500, "medium")
        .otherwise("high"),
    )
    .drop("price_history_raw", "first_price")
)

silver_products.write.mode("overwrite").format("delta").saveAsTable(SILVER_PRODUCTS)
print(f"Silver products written: {silver_products.count()}")

### Build silver_reviews

**Select & rename:** Map Cosmos DB camelCase fields (`productId`, `customerName`, `reviewDate`, `reviewText`) to snake_case silver columns.

**Validate:** Require non-null `review_id`, `product_id`, and `review_date`. Stars must be between 1 and 5 (inclusive).

**Enrich:**
- `review_year_month` (`yyyy-MM`) — useful as a partition or grouping key for time-series analysis.
- `sentiment_bucket` — maps star ratings to human-readable labels: **positive** (4–5 ★), **neutral** (3 ★), **negative** (1–2 ★).

In [ ]:
silver_reviews = (
    reviews_raw
    .select(
        F.col("id").alias("review_id"),
        F.col("productId").alias("product_id"),
        F.col("categoryName").alias("category_name"),
        F.col("customerName").alias("customer_name"),
        F.to_timestamp("reviewDate").alias("review_date"),
        F.col("stars").cast("int"),
        F.col("reviewText").alias("review_text"),
    )
    # Validate
    .filter(
        F.col("review_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("stars").between(1, 5)
        & F.col("review_date").isNotNull()
    )
    # Enrich
    .withColumn("review_year_month", F.date_format("review_date", "yyyy-MM"))
    .withColumn(
        "sentiment_bucket",
        F.when(F.col("stars") >= 4, "positive")
        .when(F.col("stars") == 3, "neutral")
        .otherwise("negative"),
    )
)

silver_reviews.write.mode("overwrite").format("delta").saveAsTable(SILVER_REVIEWS)
print(f"Silver reviews written: {silver_reviews.count()}")

### Build silver_price_history

Flatten the `priceHistory` JSON array (same `from_json` + `explode` pattern as Step 3) so that each historical price entry becomes its own row with `product_id`, `price_date`, and `price`.

**Window functions** (partitioned by `product_id`, ordered by `price_date`):
- `price_seq` — ordinal position of each price point via `row_number()`
- `price_change_pct` — percentage change from the immediately preceding price point via `lag()`. `NULL` for the first entry in each product's history.

In [ ]:
from pyspark.sql.window import Window

# Same schema used in Cell 3 — parse the JSON string before exploding
price_history_schema = "array<struct<date:string,price:double>>"

silver_price_history = (
    products_raw
    .withColumn("priceHistoryParsed", F.from_json(F.col("priceHistory"), price_history_schema))
    .select(
        F.col("id").alias("product_id"),
        F.col("name").alias("product_name"),
        F.col("categoryName").alias("category_name"),
        F.explode("priceHistoryParsed").alias("price_entry"),
    )
    .select(
        "product_id",
        "product_name",
        "category_name",
        F.to_timestamp(F.col("price_entry.date")).alias("price_date"),
        F.col("price_entry.price").cast("double").alias("price"),
    )
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
    # Add sequence number and price change from previous entry
    .withColumn(
        "price_seq",
        F.row_number().over(
            Window.partitionBy("product_id").orderBy("price_date")
        ),
    )
    .withColumn(
        "prev_price",
        F.lag("price").over(
            Window.partitionBy("product_id").orderBy("price_date")
        ),
    )
    .withColumn(
        "price_change_pct",
        F.round(
            F.when(
                F.col("prev_price").isNotNull(),
                ((F.col("price") - F.col("prev_price")) / F.col("prev_price")) * 100,
            ),
            2,
        ),
    )
    .drop("prev_price")
)

silver_price_history.write.mode("overwrite").format("delta").saveAsTable(
    SILVER_PRICE_HISTORY
)
print(f"Silver price history written: {silver_price_history.count()}")

### Summary and data-quality logging

Re-read each silver table to get final row counts and print a JSON summary.

Then record observability metadata via `metastore_functions`:
- **`log_data_quality`** — input/output/rejected counts plus named validation rules for `silver_products` and `silver_reviews`.
- **`log_dataset_profile`** — column-level null rates and min/max stats for `silver_products`.
- **`log_transform_lineage`** — source datasets, transforms applied, and columns added for all three silver tables. This powers the lineage graph in downstream dashboards.

In [ ]:
# Cell 6: Print summary, log data quality and lineage
import json

try:
    silver_products_count = spark.read.table(SILVER_PRODUCTS).count()
    silver_reviews_count = spark.read.table(SILVER_REVIEWS).count()
    silver_price_history_count = spark.read.table(SILVER_PRICE_HISTORY).count()

    summary = {
        "run_id": run_id,
        "silver_products_count": silver_products_count,
        "silver_reviews_count": silver_reviews_count,
        "silver_price_history_count": silver_price_history_count,
        "status": "success",
    }
    print(json.dumps(summary, indent=2))

    # Log data quality for silver_products
    products_raw_count = spark.read.table(BRONZE_TABLE).filter("docType = 'product'").count()
    metastore.log_data_quality(
        datasetId="silver.products", runId=run_id,
        totalInput=products_raw_count,
        totalOutput=silver_products_count,
        rejected=products_raw_count - silver_products_count,
        rulesResults=[
            {"ruleId": "S001", "name": "NonNullProductId", "passed": True},
            {"ruleId": "S002", "name": "PriceRange", "passed": True},
        ])

    # Log data quality for silver_reviews
    reviews_raw_count = spark.read.table(BRONZE_TABLE).filter("docType = 'review'").count()
    metastore.log_data_quality(
        datasetId="silver.reviews", runId=run_id,
        totalInput=reviews_raw_count,
        totalOutput=silver_reviews_count,
        rejected=reviews_raw_count - silver_reviews_count,
        rulesResults=[
            {"ruleId": "R001", "name": "NonNullReviewId", "passed": True},
            {"ruleId": "R002", "name": "StarRange", "passed": True},
        ])

    # Log dataset profiles
    metastore.log_dataset_profile(
        datasetId="silver.products", runId=run_id, rowCount=silver_products_count,
        columnProfiles=[
            {"column": "current_price", "nullRate": 0.0, "min": "varies", "max": "varies"},
            {"column": "inventory", "nullRate": 0.0},
        ],
        notes="")

    # Log transform lineage
    metastore.log_transform_lineage(
        datasetId="silver.products", runId=run_id,
        sourceDatasets=["bronze.SampleData"],
        transforms=["filter(docType='product')", "cast_types", "add_price_change_pct", "add_inventory_status"],
        columnsAdded=["price_change_pct", "price_change_direction", "inventory_status"])

    metastore.log_transform_lineage(
        datasetId="silver.reviews", runId=run_id,
        sourceDatasets=["bronze.SampleData"],
        transforms=["filter(docType='review')", "cast_types", "add_sentiment_bucket"],
        columnsAdded=["sentiment_bucket"])

    metastore.log_transform_lineage(
        datasetId="silver.price_history", runId=run_id,
        sourceDatasets=["bronze.SampleData"],
        transforms=["filter(docType='product')", "explode(priceHistory)", "cast_types"],
        columnsAdded=["price_date", "price"])

except Exception as e:
    raise